# Traductor Inglés → Español con GPT-6 (API de OpenAI)

En este cuaderno implementaremos el mismo traductor de inglés a español, pero ahora con un **modelo de lenguaje de propósito general de última generación: GPT-6 de OpenAI**, usando su API.

### La familia GPT-6 (precios por millón de tokens, entrada / salida)
| Modelo | Para qué sirve | Precio aprox. |
|---|---|---|
| `gpt-6-luna` | El más económico y rápido, ideal para tareas enfocadas y de alto volumen (como traducir) | $0.10 / $0.50 |
| `gpt-6.1-terra` | Equilibrio entre capacidad y costo | $1 / $5 |
| `gpt-6.1-sol` | Más capacidad para tareas exigentes | $2 / $10 |
| `gpt-6-astra` | El más potente, para razonamiento complejo | $10 / $50 |

Usaremos `gpt-6-luna`: para traducción sobra capacidad y cuesta una fracción.

### Diferencia con los cuadernos anteriores
- **10-04 / 10-05 (Local):** un modelo *especializado* en traducir (MarianMT, ~75M parámetros).
- **Este cuaderno:** un LLM *generalista* al que le **pedimos** traducir mediante instrucciones (prompt). Puede además adaptar el tono o el registro.

### ¿Cómo obtener la API Key de OpenAI?
1. Ir a [platform.openai.com](https://platform.openai.com) e iniciar sesión.
2. Crear una clave en **API keys** → **Create new secret key** (la API se paga por uso; revisa tu saldo y límites).
3. En Colab, ir a **Secretos** (ícono de llave 🔑) y agregar un secreto llamado `OPENAI_API_KEY` con la clave.

## Paso 1: Configurar la API Key

Cargamos la clave desde los secretos de Colab (más seguro que escribirla en el código).

In [4]:
from google.colab import userdata
import os

api_key = userdata.get('OPENAI_API_KEY')

# El SDK de OpenAI lee la clave de la variable de entorno OPENAI_API_KEY
os.environ['OPENAI_API_KEY'] = api_key

# Verificar que se haya cargado (nunca imprimas la clave, ni siquiera parcialmente)
print("API Key cargada:", "Sí" if api_key else "No")

API Key cargada: Sí


## Paso 2: Instalar el SDK de OpenAI

In [5]:
!pip install -q -U openai

## Paso 3: Crear el cliente

OpenAI recomienda la **Responses API** (`client.responses.create`) para los modelos nuevos; es la que usaremos.

In [6]:
from openai import OpenAI

# El cliente toma la clave de la variable de entorno OPENAI_API_KEY
client = OpenAI()

print("Cliente de OpenAI creado exitosamente!")

Cliente de OpenAI creado exitosamente!


## Paso 4: Traducir con GPT-6

Parámetros principales de `responses.create()`:

- `model`: el modelo a usar (`gpt-6-luna`).
- `instructions`: instrucciones de sistema (el "rol" del modelo). Aquí le decimos que es un traductor.
- `input`: el texto del usuario.
- `reasoning={"effort": "none"}`: GPT-6 es un modelo de razonamiento; para una tarea simple como traducir desactivamos el razonamiento extra para que sea más rápido y barato (también puede ser `low`, `medium`, `high`...).
- `max_output_tokens`: límite de tokens de salida.

> Los modelos de razonamiento como GPT-6 **no usan** el parámetro `temperature`; el control se hace con `reasoning` y con las instrucciones.

In [7]:
MODELO = "gpt-6-luna"   # También puedes probar "gpt-6.1-terra", "gpt-6.1-sol" o "gpt-6-astra"

INSTRUCCIONES = (
    "Eres un traductor profesional de inglés a español. "
    "Traduce el texto del usuario de forma natural y fiel. "
    "Responde únicamente con la traducción, sin comentarios."
)

texto_ingles = "Artificial intelligence is transforming industries around the world."

response = client.responses.create(
    model=MODELO,
    instructions=INSTRUCCIONES,
    input=texto_ingles,
    reasoning={"effort": "none"},
    max_output_tokens=500,
)

print("Texto original en inglés:")
print(texto_ingles)
print("\nTraducción al español:")
print(response.output_text)

Texto original en inglés:
Artificial intelligence is transforming industries around the world.

Traducción al español:
La inteligencia artificial está transformando industrias en todo el mundo.


## Paso 5: Función reutilizable para traducir

Encapsulamos la llamada en una función, igual que en los cuadernos anteriores.

In [8]:
def translate_english_to_spanish(text, modelo=MODELO):
    """
    Traduce texto de inglés a español usando GPT-6 vía la API de OpenAI.

    Args:
        text (str): Texto en inglés a traducir.
        modelo (str): Modelo GPT-6 a usar.

    Returns:
        str: Texto traducido al español.
    """
    try:
        response = client.responses.create(
            model=modelo,
            instructions=INSTRUCCIONES,
            input=text,
            reasoning={"effort": "none"},
            max_output_tokens=1000,
        )
        return response.output_text.strip()

    except Exception as e:
        return f"Error en la traducción: {str(e)}"

## Paso 6: Ejemplos prácticos

Usamos los mismos textos que en los cuadernos 10-04 y 10-05 para poder comparar los resultados.

In [9]:
english_paragraph = (
    "Artificial intelligence is transforming industries around the world. "
    "It has applications in healthcare, education, and many other fields. "
    "The potential of AI to improve lives is enormous, but it also raises ethical concerns."
)

print("Texto original en inglés:")
print(english_paragraph)
print("\nTraducción al español:")
print(translate_english_to_spanish(english_paragraph))
print("\n" + "="*70 + "\n")

frases = [
    "Machine learning is a subset of artificial intelligence.",
    "The robot can understand natural language and respond accordingly.",
    "Deep learning models require large amounts of data to train effectively."
]

for i, frase in enumerate(frases, 1):
    print(f"Frase {i}:")
    print(f"  EN: {frase}")
    print(f"  ES: {translate_english_to_spanish(frase)}")
    print()

Texto original en inglés:
Artificial intelligence is transforming industries around the world. It has applications in healthcare, education, and many other fields. The potential of AI to improve lives is enormous, but it also raises ethical concerns.

Traducción al español:
La inteligencia artificial está transformando industrias de todo el mundo. Tiene aplicaciones en la atención sanitaria, la educación y muchos otros campos. Su potencial para mejorar la vida de las personas es enorme, pero también plantea preocupaciones éticas.


Frase 1:
  EN: Machine learning is a subset of artificial intelligence.
  ES: El aprendizaje automático es una rama de la inteligencia artificial.

Frase 2:
  EN: The robot can understand natural language and respond accordingly.
  ES: El robot puede entender el lenguaje natural y responder en consecuencia.

Frase 3:
  EN: Deep learning models require large amounts of data to train effectively.
  ES: Los modelos de aprendizaje profundo necesitan grandes cant

## Paso 7: Algo que un modelo especializado no hace — controlar el tono

Como GPT-6 sigue instrucciones, podemos pedir un registro específico. Cambia las instrucciones y observa cómo cambia la traducción.

In [10]:
frase = "Hey, could you send me the report by tomorrow? Thanks a lot!"

for registro in ["muy formal (usted)", "informal y cercano (tú)"]:
    response = client.responses.create(
        model=MODELO,
        instructions=f"Traduce del inglés al español con un registro {registro}. Responde solo con la traducción.",
        input=frase,
        reasoning={"effort": "none"},
        max_output_tokens=300,
    )
    print(f"[{registro}] -> {response.output_text.strip()}")

[muy formal (usted)] -> ¿Podría enviarme el informe a más tardar mañana? ¡Muchas gracias!
[informal y cercano (tú)] -> Oye, ¿podrías enviarme el informe para mañana? ¡Muchas gracias!


## Conclusión

- Un modelo **especializado** (MarianMT) es pequeño, gratuito y funciona offline, pero hace solo una cosa.
- Un **LLM generalista** (GPT-6) traduce con más naturalidad, entiende el contexto y permite controlar el tono, pero requiere API Key y se paga por uso.
- Con `gpt-6-luna` el costo de traducir es muy bajo (≈ $0.10 por millón de tokens de entrada).